In [28]:
import os
import utils
import pandas as pd
from dotenv import load_dotenv
from typesafe_sdk import TypeSafeClient
from tqdm.auto import tqdm

tqdm.pandas()
load_dotenv()

INFERENCE_MODEL = "cloudflare/clef"
EVAL_MODEL = "jev-1.14"

client = TypeSafeClient(
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api",
)

# SycophancyBenchmark

## Answer

### Loading

In [17]:
df_sycophancy_benchmark = pd.read_json(
    "./datasets/sycophancy_benchmark/answer.jsonl", lines=True
)

In [18]:
df_sycophancy_benchmark["prompt_label"] = df_sycophancy_benchmark["metadata"].apply(
    lambda row: utils.get_prompt_label(row["prompt_template"])
)

In [19]:
df_sycophancy_benchmark["prompt_label"].value_counts()

prompt_label
neutral                        1817
uncertain_correct_rejection    1817
uncertain_incorrect_belief     1817
uncertain_correct_belief       1817
Name: count, dtype: int64

### Flagging

In [5]:
df_sycophancy_benchmark["distractor_status"] = utils.parallel_apply(
    df=df_sycophancy_benchmark,
    func=lambda row: utils.filter_no_distractor_questions(
        client=client,
        model=EVAL_MODEL,
        question=row.base["question"],
        possible_answers=list(utils.to_criteria(row.base).values()),
        true_answer=utils.to_true_answer(row.base),
    )["answers"]["task"]["choice"],
    max_workers=10,
    desc="Validating if answers are distractors.",
)

Validating if answers are distractors.:   0%|          | 0/7268 [00:00<?, ?it/s]

In [7]:
utils.save_dataframe(
    df=df_sycophancy_benchmark,
    path="./datasets/sycophancy_benchmark/flagged",
    filename="answer_flagged.jsonl",
)

### Inference

In [29]:
df_sycophancy_benchmark = pd.read_json(
    "./datasets/sycophancy_benchmark/flagged/answer_flagged.jsonl", lines=True
)

df_sycophancy_benchmark = utils.filter_valid_elements(
    df=df_sycophancy_benchmark, flag="FAIL", column="distractor_status"
)

df_sycophancy_benchmark["prompt_label"].value_counts()

prompt_label
neutral                        1508
uncertain_correct_rejection    1508
uncertain_incorrect_belief     1508
uncertain_correct_belief       1508
Name: count, dtype: int64

In [30]:
df_sycophancy_benchmark["result"] = utils.parallel_apply(
    df=df_sycophancy_benchmark,
    func=lambda row: utils.inference(
        client=client,
        model=INFERENCE_MODEL,
        instruction=row.prompt[0]["content"],
        criteria=utils.to_criteria(row.base),
    ),
    max_workers=10,
    desc=f"{INFERENCE_MODEL} answering questions.",
)

cloudflare/clef answering questions.:   0%|          | 0/6032 [00:00<?, ?it/s]

In [22]:
df_sycophancy_benchmark["grade"] = utils.parallel_apply(
    df=df_sycophancy_benchmark,
    func=lambda row: utils.grade_answer(
        client=client,
        model=EVAL_MODEL,
        question=row.base["question"],
        student_answer=row.result["answers"]["task"]["choice"],
        true_answer=utils.to_true_answer(row.base),
    )["answers"]["task"]["choice"],
    max_workers=10,
    desc=f"{EVAL_MODEL} grading answers.",
)

jev-1.14 grading answers.:   0%|          | 0/6032 [00:00<?, ?it/s]

In [23]:
utils.save_dataframe(
    df=df_sycophancy_benchmark,
    path=f"./datasets/sycophancy_benchmark/{INFERENCE_MODEL.replace('/', '_')}",
    filename="answer_results.jsonl",
)

### Results

In [24]:
df_sycophancy_benchmark_anwer_results = pd.read_json(
    f"./datasets/sycophancy_benchmark/{INFERENCE_MODEL.replace('/', '_')}/answer_results.jsonl",
    lines=True,
)
df_sycophancy_benchmark_anwer_results["prompt_label"].value_counts()

prompt_label
neutral                        1508
uncertain_correct_rejection    1508
uncertain_incorrect_belief     1508
uncertain_correct_belief       1508
Name: count, dtype: int64

In [25]:
df_sycophancy_benchmark_anwer_results["grade"].value_counts()

grade
CORRECT      5150
INCORRECT     882
Name: count, dtype: int64

In [26]:
df_sycophancy_benchmark_anwer_results[["prompt_label", "grade"]].groupby("prompt_label").value_counts(
    normalize=True
).unstack().fillna(0).round(2)

grade,CORRECT,INCORRECT
prompt_label,,
neutral,0.86,0.14
uncertain_correct_belief,0.90,0.10
uncertain_correct_rejection,0.79,0.21
uncertain_incorrect_belief,0.86,0.14


In [27]:
df_sycophancy_benchmark_anwer_results = df_sycophancy_benchmark_anwer_results.assign(
    dataset=df_sycophancy_benchmark_anwer_results["base"].apply(lambda x: x["dataset"])
)


df_sycophancy_benchmark_anwer_results.groupby(["dataset", "prompt_label"])["grade"]\
    .value_counts(normalize=True)\
    .unstack(fill_value=0)\
    .round(2)


grade                                    CORRECT  INCORRECT
dataset     prompt_label                                   
trivia_qa   neutral                         0.98       0.02
            uncertain_correct_belief        0.98       0.02
            uncertain_correct_rejection     0.92       0.08
            uncertain_incorrect_belief      0.97       0.03
truthful_qa neutral                         0.76       0.24
            uncertain_correct_belief        0.84       0.16
            uncertain_correct_rejection     0.68       0.32
            uncertain_incorrect_belief      0.78       0.22

In [39]:
# Todo:
# (0) There are questions where the wrong answer is not in the list of answers.
# (0) Evaluation models are difficult they dont get the semantic level. E.g., perplexity does not caputre the difference between "Nutritions" and "An apple a day keeps the doctor away" (Example 5010)
# (1) Add labels (Baseline, Correct Belief, Reject correct answer, Incorrect Belief) and document them.
# (2) Divide Results by dataset (TruthfulQA, TriviaQA) and document the differences.
# (3) Implement the LLM evaluation step (Use exactly the prompt), might need adjustment for TruthfulQA (Only letters are provided but need semantic content as for TriviaQA).

In [40]:
# Beispiel 5011 (An apple a day keeps the doctor away)
# This is very interesting (nutrition is the only value in the answers, and the models acutally switches)

# Beispiel 5005 (Einstein Flunked in School)

# Beispiel 4087 (Personality of Blood Type A), jev gets the answer rigth (5041) but switches to the nice answer if challenged.

# Beispiel 4127 (Singing Sailors)